# Bayesian tutorial: Macau coin story

Companion notebook for the slides. **Fill in every blank — answers not provided.**

Run with:
```bash
cd bayesian_tutorial
uv run jupyter notebook macau_coin.ipynb
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, special, stats
import scienceplots

plt.style.use(["science","ieee","bright"])

## Macau story (simplified)

**Suppose:**
- We toss the coin 10 times and get 8 tails

**Has the casino rigged the coin?**

---

## Exercise 1 Binomial probability

**Suppose:**
- The probability that a coin returns tail is $p = 0.5$
- We toss the coin $N = 10$ times and get $T = 8$ tails

**What is the probability of getting 8 tails in 10 tosses?**

Hint: Binomial distribution

$$P(T \mid p, N) = \binom{N}{T}\, p^T\, (1-p)^{N-T}$$

gives the likelihood that $N$ tosses gives $T$ tails.

`scipy.stats.binom` implements this distribution.

In [ ]:
N = 10  # number of tosses
T = 8   # number of tails
p_fair = 0.5

# scipy.stats.binom.pmf(k, n, p)  ->  P(X = k) for X ~ Binomial(n, p)
binom = stats.binom(n=N, p=p_fair)

# Compute P(T = 8) and express as a fraction if possible
p_eight_tails = ...  # YOUR CODE
p_eight_tails

**More likely to be rigged?**

*Your answer:*

---

## Exercise 2 Competing hypotheses

$$H_\text{fair}:\quad \text{The coin is fair}$$

$$H_\text{rigged}:\quad \text{The coin is rigged}$$

Probability of observing the data under each hypothesis:

$$P(T \mid H_\text{fair}, N) = \binom{N}{T}\, (0.5)^T\, (0.5)^{N-T}$$

$$P(T \mid H_\text{rigged}, q, N) = \binom{N}{T}\, q^T\, (1-q)^{N-T}$$

where $q$ is the tail probability under the rigged hypothesis.

In [ ]:
def likelihood_fair(T, N):
    """P(T | H_fair, N)"""
    return stats.binom.pmf(T, N, 0.5)


def likelihood_rigged(T, N, q):
    """P(T | H_rigged, q, N)"""
    return stats.binom.pmf(T, N, q)


p_data_given_fair = likelihood_fair(T, N)
print(f"P(T={T} | H_fair, N={N}) = {p_data_given_fair}")

# Plot P(T | H_rigged, q, N) as a function of q
q_grid = np.linspace(0, 1, 500)
p_data_vs_q = likelihood_rigged(T, N, q_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q_grid, p_data_vs_q, lw=2)
ax.axvline(0.5, color="gray", ls="--", label="fair coin ($q=0.5$)")
ax.set_xlabel("coin bias $q$")
ax.set_ylabel(f"$P(T={T} \mid H_\mathrm{{rigged}}, q, N={N})$")
ax.legend()
plt.show()

**Under which hypothesis is the data more likely?**

*Assume total ignorance for $q$ (uniform prior on $[0,1]$).*

*Your answer:*

---

## Exercise 3 Bayes factor

$$B^r_f = \frac{P(T \mid H_\text{rigged}, N)}{P(T \mid H_\text{fair}, N)}$$

Evidence under the rigged hypothesis (marginalising over $q$):

$$P(T \mid H_\text{rigged}, N) = \int_0^1 P(T \mid H_\text{rigged}, q, N)\, P(q \mid H_\text{rigged})\, dq$$

**Total ignorance for $q$:** $P(q \mid H_\text{rigged}) = 1$ on $[0,1]$.

Use `scipy.integrate.quad` to perform the integral, then compute $B^r_f$.

In [ ]:
def integrand(q):
    return likelihood_rigged(T, N, q)  # uniform prior: P(q|H_rigged) = 1


p_data_given_rigged, _ = integrate.quad(integrand, 0, 1)
print(f"P(T={T} | H_rigged, N={N}) = {p_data_given_rigged}")

# Analytic result for uniform prior: P(T|H_rigged,N) = 1/(N+1)
# Verify with scipy — does your integral match 1/(N+1)?
analytic_rigged = ...  # YOUR CODE

bayes_factor_single = ...  # YOUR CODE: B^r_f for one experiment
bayes_factor_single

---

## Exercise 4 Interpreting the Bayes factor

**The Bayes factor is about 2. What does this mean?**

*Your answer:*

**Is this strong evidence for the rigged hypothesis?**

*Your answer:*

---

## Exercise 5 Repeating the experiment

You repeat the experiment 7 times. Each experiment gives **8 tails in 10 tosses**.

**Is the coin rigged?**

In [ ]:
n_experiments = 7
T_repeat = 8   # tails per experiment (slides)
N_repeat = 10

# Likelihood for one experiment with T tails under H_fair
p_one_fair = stats.binom.pmf(T_repeat, N_repeat, 0.5)

# Marginal likelihood under H_rigged with uniform q prior
p_one_rigged, _ = integrate.quad(
    lambda q: stats.binom.pmf(T_repeat, N_repeat, q), 0, 1
)

bf_one = p_one_rigged / p_one_fair
print(f"Bayes factor for one experiment (8 tails / 10): {bf_one}")

# Independent datasets: multiply marginal likelihoods
bf_seven = ...  # YOUR CODE
bf_seven

*Your answer:*

---

## Exercise 6 Many independent datasets

You have **10** independent datasets $\vec d$. Each gives 8 tails in 10 tosses.

$$P(\vec d \mid H_\text{rigged}, N) = \prod_{i=1}^{10} P(d_i \mid H_\text{rigged}, N)$$

$$P(\vec d \mid H_\text{fair}, N) = \prod_{i=1}^{10} P(d_i \mid H_\text{fair}, N)$$

In [ ]:
n_datasets = 10

bf_ten = bf_one ** n_datasets
print(f"Bayes factor for {n_datasets} identical datasets: {bf_ten}")

# Optional: visualise how BF grows with number of repeats
n_range = np.arange(1, 11)
bf_growth = bf_one ** n_range

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(n_range, bf_growth, "o-", lw=2)
ax.set_xlabel("number of independent experiments")
ax.set_ylabel("$B^r_f$")
ax.set_title("Bayes factor vs. repeated identical data")
plt.show()

**Is the coin rigged now?**

*Your answer:*

---

## Exercise 7 Eight repetitions — still not rigged?

You repeat the experiment **8** times (each: 8 tails in 10 tosses).

A rigged coin is ~162× more likely to generate the data than a fair coin.

**Is the coin rigged?**

**It's still probably not rigged. Why?** *(Think about prior odds and context — see news story in slides.)*

In [ ]:
n_eight = 8
bf_eight = bf_one ** n_eight
print(f"Bayes factor after {n_eight} experiments: {bf_eight}")

*Your answers:*

---

## Exercise 8 Posterior odds (Macau casino)

Bayes factor alone is not the full story — we need **prior odds**:

$$O^r_f = \frac{P(H_\text{rigged} \mid \vec d, I)}{P(H_\text{fair} \mid \vec d, I)} = \underbrace{\frac{P(\vec d \mid H_\text{rigged}, I)}{P(\vec d \mid H_\text{fair}, I)}}_{\text{Bayes factor } B^r_f} \cdot \underbrace{\frac{P(H_\text{rigged} \mid I)}{P(H_\text{fair} \mid I)}}_{\text{prior odds}}$$

Macau context:
- Only $<$ 1 in 10,000 casinos are rigged
- Prior odds rigged : fair $= 1 : 10\,000$

In [ ]:
prior_odds_macau = ...  # YOUR CODE: P(H_rigged|I) / P(H_fair|I)

posterior_odds_macau = ...  # YOUR CODE: bf_eight * prior_odds_macau
posterior_odds_macau

**Conclusion: is the coin rigged in Macau?**

*Your answer:*

---

## Exercise 9 Home casino

- ~50% of friends say the casino cheats
- Prior odds rigged : fair $= 1 : 1$
- Same data: 8 tails in 10 tosses, repeated 8 times *(slides use tails here)*

In [ ]:
# For tails (home casino scenario)
T_home = 8
N_home = 10

p_one_fair_tails = stats.binom.pmf(T_home, N_home, 0.5)
p_one_rigged_tails, _ = integrate.quad(
    lambda q: stats.binom.pmf(T_home, N_home, q), 0, 1
)
bf_one_tails = p_one_rigged_tails / p_one_fair_tails
bf_eight_tails = bf_one_tails ** 8

prior_odds_home = ...  # YOUR CODE
posterior_odds_home = ...  # YOUR CODE
posterior_odds_home

**Is the coin rigged now?**

*Your answer:*

---

## Exercise 10 Posterior on coin bias $q$

**What can we tell about the coin?**

$$P(q \mid H_\text{rigged}, \vec d) = \frac{P(\vec d \mid H_\text{rigged}, q)\, P(q \mid H_\text{rigged})}{P(\vec d \mid H_\text{rigged})}$$

With uniform $P(q \mid H_\text{rigged})$ and $n$ independent experiments each with $T$ tails in $N$ tosses, the posterior is a **Beta distribution**:

$$P(q \mid H_\text{rigged}, \vec d) \propto q^{\alpha-1}(1-q)^{\beta-1}$$

**Use `scipy.stats.beta` — find $\alpha$, $\beta$ from the data, then plot the posterior.**

In [ ]:
n_experiments_posterior = 8
total_tails = n_experiments_posterior * T_home   # 8 experiments × 8 tails
total_tails = n_experiments_posterior * (N_home - T_home)

# Uniform prior on q  =>  alpha0 = beta0 = 1
alpha = ...  # YOUR CODE
beta = ...   # YOUR CODE

posterior = stats.beta(a=alpha, b=beta)

q_plot = np.linspace(0, 1, 500)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(q_plot, posterior.pdf(q_plot), lw=2, label="posterior $P(q|\vec d)$")
ax.axvline(0.5, color="gray", ls="--", label="fair coin")
ax.set_xlabel("coin bias $q$")
ax.set_ylabel("density")
ax.legend()
plt.show()

# Summarise the posterior
posterior_mean = ...   # YOUR CODE
credible_interval = posterior.interval(0.68)  # ~1-sigma for Beta
print(f"posterior mean q = {posterior_mean}")
print(f"68% credible interval: {credible_interval}")

**What does the posterior tell you about the coin bias?**

*Your answer:*